# Multi-GNN Tutorial - Google Colab (Usando env.yml)

Este notebook usa el archivo env.yml del repositorio Multi-GNN para crear el entorno exacto necesario.

## 1. Configuración del Entorno con Conda

In [ ]:
# Instalar conda en Google Colab
!pip install -q condacolab
import condacolab
condacolab.install()

**IMPORTANTE: Después de ejecutar la celda anterior, el runtime se reiniciará automáticamente. Continúa con las siguientes celdas después del reinicio.**

In [ ]:
# Clonar el repositorio Multi-GNN
!git clone https://github.com/IBM/Multi-GNN.git
%cd Multi-GNN

In [ ]:
# Verificar que el env.yml existe
!cat env.yml

In [ ]:
# Crear el entorno usando el env.yml
# Nota: Adaptamos para CPU ya que Colab gratuito usa CPU por defecto
!conda env create -f env.yml

In [ ]:
# Verificar que el entorno se creó correctamente
!conda env list

## 2. Subir el Dataset de Kaggle

In [ ]:
from google.colab import files
import os

# Crear directorio para los datos
!mkdir -p data/kaggle

print("Por favor, sube el archivo HI-Small_Trans.csv")
uploaded = files.upload()

# Mover el archivo a la carpeta data/kaggle
for filename in uploaded.keys():
    !mv {filename} data/kaggle/{filename}
    print(f"Archivo {filename} movido a data/kaggle/")

## 3. Formatear los Datos usando el código del repositorio

In [ ]:
# Ejecutar el script de formateo del repositorio
# Este script usa el env.yml y todas las dependencias correctas
!conda run -n multignn python format_kaggle_files.py \
    --csvPath data/kaggle/HI-Small_Trans.csv \
    --outPath data/kaggle/formatted_transactions.csv

## 4. Entrenar el modelo GINe

Ahora usamos el script de entrenamiento del repositorio con el entorno conda configurado.

In [ ]:
# Entrenar el modelo GINe usando el código del repositorio
!conda run -n multignn python training.py \
    --dataPath data/kaggle/formatted_transactions.csv \
    --model GINe \
    --epochs 50 \
    --hidden_dim 64 \
    --num_layers 3 \
    --batch_size 512 \
    --lr 0.001 \
    --device cpu

## 5. (Opcional) Entrenar otros modelos

In [ ]:
# Entrenar GATe
!conda run -n multignn python training.py \
    --dataPath data/kaggle/formatted_transactions.csv \
    --model GATe \
    --epochs 50 \
    --hidden_dim 64 \
    --num_layers 3 \
    --batch_size 512 \
    --lr 0.001 \
    --device cpu

In [ ]:
# Entrenar PNA
!conda run -n multignn python training.py \
    --dataPath data/kaggle/formatted_transactions.csv \
    --model PNA \
    --epochs 50 \
    --hidden_dim 64 \
    --num_layers 3 \
    --batch_size 512 \
    --lr 0.001 \
    --device cpu

## 6. (Opcional) Código Interactivo - Ejecutar paso a paso

Si prefieres ejecutar el código paso a paso de forma interactiva:

In [ ]:
# Activar el entorno conda en el kernel actual
import sys
sys.path.insert(0, '/usr/local/envs/multignn/lib/python3.9/site-packages')

# Ahora puedes importar los módulos del repositorio
from data_loading import get_data
from models import GINe
from training import train_homo, evaluate_homo
from train_util import get_loaders
import torch
from types import SimpleNamespace

print("Módulos importados correctamente")

In [ ]:
# Configurar argumentos
args = SimpleNamespace(
    dataPath='data/kaggle/formatted_transactions.csv',
    device='cpu',
    hidden_dim=64,
    num_layers=3,
    batch_size=512,
    num_neighbors=[10, 5],
    lr=0.001,
    epochs=50
)

# Cargar datos
print("Cargando datos...")
tr_data, val_data, te_data, tr_inds, val_inds, te_inds = get_data(args)
print(f"Datos cargados: {tr_data.num_nodes} nodos, {tr_data.num_edges} edges")

In [ ]:
# Crear dataloaders
print("Creando dataloaders...")
tr_loader, val_loader, te_loader = get_loaders(
    tr_data, val_data, te_data,
    tr_inds, val_inds, te_inds,
    args
)
print("Dataloaders creados")

In [ ]:
# Crear modelo
print("Creando modelo GINe...")
model = GINe(
    in_dim=tr_data.x.size(1),
    edge_dim=tr_data.edge_attr.size(1),
    hidden_dim=args.hidden_dim,
    num_layers=args.num_layers
).to(args.device)

optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
print(f"Modelo creado con {sum(p.numel() for p in model.parameters())} parámetros")

In [ ]:
# Calcular pesos de clase para balanceo
num_licit = (tr_data.edge_label == 0).sum().item()
num_illicit = (tr_data.edge_label == 1).sum().item()
total = num_licit + num_illicit
weight_licit = total / (2 * num_licit)
weight_illicit = total / (2 * num_illicit)
class_weights = torch.tensor([weight_licit, weight_illicit]).to(args.device)

print(f"Clases: Lícitas={num_licit}, Ilícitas={num_illicit}")
print(f"Pesos: Lícitas={weight_licit:.4f}, Ilícitas={weight_illicit:.4f}")

In [ ]:
# Entrenar
print("Iniciando entrenamiento...")
for epoch in range(args.epochs):
    loss = train_homo(model, tr_loader, optimizer, args.device, class_weights)
    
    if (epoch + 1) % 10 == 0:
        train_metrics = evaluate_homo(model, tr_loader, args.device)
        val_metrics = evaluate_homo(model, val_loader, args.device)
        
        print(f"Epoch {epoch+1}/{args.epochs}:")
        print(f"  Loss: {loss:.4f}")
        print(f"  Train F1: {train_metrics['f1']:.4f}, Acc: {train_metrics['acc']:.4f}")
        print(f"  Val F1: {val_metrics['f1']:.4f}, Acc: {val_metrics['acc']:.4f}")

print("\nEntrenamiento completado")

In [ ]:
# Evaluar en test set
print("Evaluando en test set...")
test_metrics = evaluate_homo(model, te_loader, args.device)

print("\nResultados finales en Test:")
print(f"  F1-Score: {test_metrics['f1']:.4f}")
print(f"  Accuracy: {test_metrics['acc']:.4f}")
print(f"  Precision: {test_metrics['precision']:.4f}")
print(f"  Recall: {test_metrics['recall']:.4f}")